<a href="https://colab.research.google.com/github/rstoiberer/week4-GroupD-/blob/main/1_st445_week4_data_cleaning_lab_anissa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ST445 Week 4 seminar: the cleaning challenge

**Group lab · Data Preprocessing**

Today's lab is a group-based gamification with three rounds: 1. coding (data cleaning), 2. implementation review, and 3. debate. You will work in groups of ~4 members.

For the first round, your group will receive a messy country–year dataset. You will need to preprocess it based on what was discussed in the lecture, and record **every decision** in the Markdown log with a **one-line justification** after each challenge, called here checkpoint.

Then, in the second round, you swap your notebook with another group through GitHub and review their decisions. You will score their implementation as well.

For the third round, groups who disagree in the chosen decisions, can defend their approach to the entire class.

The group with the highest score wins the prize; a second prize (if available) goes to the **best argument** at the contested checkpoint.

### How the session runs

| Round | What happens | Time |
|---|---|---|
| Set-up | Choose your group name, fill in members, open this notebook from your group repository | 5 min |
| **1. Cleaning** | Checkpoints 1 to 9, rotating driver and navigator at every checkpoint | 30 min |
| Hand-in | Commit and push; open a pull request; request a review from your reviewing group | 5 min |
| **2. Review** | Review another group's notebook: re-run it, score their log, hunt for problems they missed | 10 min |
| **3. Debate** | One minute per group on a contested checkpoint; class vote for the best argument | 8 min |
| Results | Provisional leaderboard; final scores after the tutor's moderation | 2 min |

### Roles: driver and navigator

* The **driver** types. Nobody else touches the keyboard.
* The **navigators** read the checklist, decide what to look at, and write the decision log for that checkpoint.
* **Rotate at every checkpoint.** The cell below prints your rotation. Everyone drives at least twice; the log records who drove.

### Scoring (100 points)

| Component | Points | How it is scored |
|---|---|---|
| **Cleaning accuracy** | 50 | Automatically, by the tutor: your saved clean data are compared with a hidden answer key, checkpoint by checkpoint |
| **Decision log** | 30 | By your reviewing group, moderated by the tutor: 2 points per checkpoint for a complete entry (18), and 12 for the argument at the contested checkpoint |
| **Your review of another group** | 20 | **Bug bounty:** 3 points per problem you find that they missed, confirmed by the answer key (up to 12); 4 for re-running their notebook and reporting the result; 4 for at least three specific, constructive comments |

Note that the first components gives your group points for your implementation. Meanwhile, the third component gives your group points for your review of the implementation of another group.

**Fair play.** Both answers at the contested checkpoint score the same: you are scored on the **argument**, not on the choice. Reviewers cannot change another group's accuracy score; they can only find what was missed. No sharing code between groups during round 1.

## Set-up

In [53]:
import hashlib, json, unicodedata
from pathlib import Path
import numpy as np
import pandas as pd


# 🟢 Fill in your group, then run
GROUP = "Group D"                              # choose a name for your group, e.g. "otters": letters only, used in file names
MEMBERS = ["Anissa", "Kay", "Richard", "Javier"]  # first names, in the order you will rotate

def check(condition, hint="Not yet."):
    print("✅" if condition else f"❌ {hint}")

CHECKPOINTS = ["Load and verify", "Structure and types", "Text and categories", "Dates", "Duplicates",
               "Units and scales", "Impossible and implausible values", "Missing values (contested)", "Validate and save"]
for i, cp in enumerate(CHECKPOINTS):
    print(f"{i + 1}. {cp:<36} driver: {MEMBERS[i % len(MEMBERS)]}")

1. Load and verify                      driver: Anissa
2. Structure and types                  driver: Kay
3. Text and categories                  driver: Richard
4. Dates                                driver: Javier
5. Duplicates                           driver: Anissa
6. Units and scales                     driver: Kay
7. Impossible and implausible values    driver: Richard
8. Missing values (contested)           driver: Javier
9. Validate and save                    driver: Anissa


Note: you can also divide the challenges (checkpoints) in between the group members.

### Round 1: Description of Challenges (Checkpoints)

| # | Checkpoint | The question it answers |
|---|---|---|
| 1 | Load and verify | Is this the file the source published, all of it? |
| 2 | Structure and types | Is every row an observation, and every column the right type? |
| 3 | Text and categories | Is each thing written one way only? |
| 4 | Dates | Is every date parsed, in one format, in UTC? |
| 5 | Duplicates | Does each country–year appear exactly once, and the right version? |
| 6 | Units and scales | Is each column in one unit? |
| 7 | Impossible and implausible values | Is every value possible, and plausible in context? |
| 8 | Missing values: **the contested checkpoint** | What is missing, why, and what do we do about it? |
| 9 | Validate and save | Does the clean data pass checks that fail loudly, and is it saved reproducibly? |

**The order matters.** Problems that create missing values (types, units, impossible values) come before deciding what to do about missing values.

**Checkpoint 1 is fully coded:** read it as a solution model, including its decision log. **Checkpoints 2 and 3** are skeletons, i.e., part of the code is provided and you must fill the blanks: `____`. **Checkpoints 4 to 9** are prompts only, so you have to write the code.

---
## Checkpoint 1: load and verify against the source  🟢 fully coded

Load the data from the ST445 GitHub. The publisher's file header says what it contains. Before changing anything, check that you have exactly that file, all of it.

**Professional habit:** read every column **as text** (`dtype=str`) and keep empty cells as empty strings (`keep_default_na=False`). Nothing is converted or hidden before we have looked, so we see every missing-value marker and every badly formatted number exactly as published.

In [54]:
# 🟢 Get the file: from the course repository, or upload it
DATA_URL = "https://raw.githubusercontent.com/LeonardooAlves/LSE-ST445/refs/heads/main/Week%204/data/week4_country_panel_raw.csv"
PUBLISHED_SHA256 = "116c3194708b7ac967189ddd29a06affc3e7b1071f4ecee10a17e3551ccb9f28"
PATH = Path("week4_country_panel_raw.csv")

if not PATH.exists():
    try:
        import urllib.request
        urllib.request.urlretrieve(DATA_URL, PATH)
    except Exception as err:
        print("Download failed:", type(err).__name__, "- upload the file with the Files panel, then run this cell again.")

print(PATH.read_text(encoding="utf-8").splitlines()[:4])

['# Source: ST445 teaching panel (synthetic, based loosely on published magnitudes)', '# Units: population in millions; GDP per person in current US dollars; health spending as % of GDP', '# Published: 125 country-years (25 countries, 2018-2022), plus 2 revised records', 'country,iso3,year,population_m,gdp_pc_usd,health_spend_pct,life_expectancy,income_group,survey_date']


**What you should see:** three comment lines starting with `#` (the source, the units, and **"Published: 125 country-years ... plus 2 revised records"**), then the header row. That published count is what we verify against.

In [55]:
# 🟢 Verify: checksum, row count, schema
digest = hashlib.sha256(PATH.read_bytes()).hexdigest()
check(digest == PUBLISHED_SHA256, "The file is not the published one: download it again.") # Verify the checksum

raw = pd.read_csv(PATH, comment="#", dtype=str, keep_default_na=False)
EXPECTED_COLUMNS = ["country", "iso3", "year", "population_m", "gdp_pc_usd", "health_spend_pct",
                    "life_expectancy", "income_group", "survey_date"]
check(list(raw.columns) == EXPECTED_COLUMNS, f"Unexpected columns: {list(raw.columns)}") # Verify the number of columns

PUBLISHED_ROWS = 125 + 2 # As mentioned in the metadata in previous cell
print(f"Rows read: {len(raw)} | published: {PUBLISHED_ROWS} | difference: {len(raw) - PUBLISHED_ROWS}")

✅
✅
Rows read: 131 | published: 127 | difference: 4


**What you should see:** ✅ for the checksum and the columns, then **131 rows read against 127 published: 4 more than the source says.** We do not fix that here. We record it and find out why in later checkpoints (look at the last row of the file, and think about duplicates).

In [56]:
# 🟢 Look before changing anything: what is in each column that should be numeric?
for col in ["year", "population_m", "gdp_pc_usd", "health_spend_pct", "life_expectancy"]:
    as_number = pd.to_numeric(raw[col], errors="coerce")
    not_numbers = raw.loc[as_number.isna(), col].value_counts()
    print(f"\n{col}: {len(not_numbers)} kinds of non-number")
    print(not_numbers.head(6).to_dict())
    print("range of the values that are numbers:", as_number.min(), "to", as_number.max())

raw.tail(3)


year: 1 kinds of non-number
{'': 1}
range of the values that are numbers: 2018.0 to 2022.0

population_m: 1 kinds of non-number
{'': 1}
range of the values that are numbers: -999.0 to 85400.0

gdp_pc_usd: 23 kinds of non-number
{'3,990': 2, '10,840': 1, '64,920': 1, '2,300': 1, '9,420': 1, '1,630': 1}
range of the values that are numbers: -10220.0 to 72360.0

health_spend_pct: 3 kinds of non-number
{'': 15, 'n/a': 11, '..': 5}
range of the values that are numbers: 2.3 to 17.8

life_expectancy: 2 kinds of non-number
{'n/a': 1, '': 1}
range of the values that are numbers: -1.0 to 824.0


,country,iso3,year,population_m,gdp_pc_usd,health_spend_pct,life_expectancy,income_group,survey_date
128,Bangladesh,BGD,2022,167.5,1800,2.4,72.1,Lower middle income,2023-03-25
129,Pakistan,PAK,2022,223.5,"1,690",3.4,65.7,Lower middle income,2023-03-24
130,Total,,,,,,,,


**What you should see:** a list of everything that is not a plain number, column by column: empty strings, `n/a`, `..`, and numbers written with thousands separators such as `"1,690"`. The ranges already hint at later checkpoints: a population of **−999**, a population in the tens of thousands, a life expectancy of several hundred, and a negative GDP. The last row of the file is a **"Total"** row, not an observation.

### ✍️ Decision log, checkpoint 1: load and verify

**Driver:** (name) · **Navigator(s):** (names)

| What we found (with a count) | What we decided | Justification (one line) |
|---|---|---|
| Checksum matches the published SHA-256 | Use this file; read every column as text | The file is the one published, and reading as text hides nothing before we look |
| 131 rows against 127 published (4 extra) | Keep all rows for now; explain the extra 4 in checkpoints 2 and 5 | Deleting before we know why would hide the cause |
| Non-numbers in numeric columns: `""`, `n/a`, `..`, thousands separators; a `-999` population; a "Total" row | Carry the list forward to checkpoints 2, 7 and 8 | Each needs a different treatment, decided where it belongs |

*This is a model entry for the decision log: every row show invclude a count where applicable, a decision, and one line of justification.*

---
## Checkpoint 2: structure and types  🟡 skeleton

Complete the challenges as indicated in the comments. The goal is to make every row an observation and every column the right type. Use what checkpoint 1 found. Fill each `____`.

In [57]:
raw

,country,iso3,year,population_m,gdp_pc_usd,health_spend_pct,life_expectancy,income_group,survey_date
0,Japan,JPN,2021,129.0,40.48,11.1,83.3,High income,2022-03-03
1,Bangladesh,BGD,2019,162.8,1700,2.3,73.0,Lower middle income,2020-03-03
2,Uganda,UGA,2018,42.7,770,3.8,63.4,low income,2019-03-06
3,Germany,DEU,2021,85300,49390,11.9,80.1,High income,2022-03-16
4,Mexico,MEX,2021,127.8,-10220,5.7,74.1,Upper middle income,2022-03-09
...,...,...,...,...,...,...,...,...,...
126,Uganda,UGA,2019,42.7,800,n/a,-1,low income,2020-03-07
127,Qatar,QAT,2019,2.8,"69,390",2.6,80.3,High income,2020-03-24
128,Bangladesh,BGD,2022,167.5,1800,2.4,72.1,Lower middle income,2023-03-25
129,Pakistan,PAK,2022,223.5,"1,690",3.4,65.7,Lower middle income,2023-03-24


In [58]:
df = raw.copy()

# (a) Remove rows that are not observations (checkpoint 1 showed one at the end of the file)
df = df[df["country"] != "Total"]

# (b) Strip leading and trailing spaces from every text value
df = df.apply(lambda s: s.str.strip())

# (c) Declare every missing-value marker checkpoint 1 found, and replace them with a real missing value
MARKERS = ["..", "n/a", "-1", "-999", ""]          # hint: three more; one of them is a number
df = df.replace({m: pd.NA for m in MARKERS})

# (d) Convert the numeric columns. GDP is written with thousands separators: remove them first
missing_before = df.isna().sum().sum()
df["year"] = df["year"].astype("Int16")          # a small nullable integer, e.g. "Int16"
df["gdp_pc_usd"] = pd.to_numeric(df["gdp_pc_usd"].str.replace(",", "", regex=False), errors="coerce")
for col in ["population_m", "health_spend_pct", "life_expectancy"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
created = df.isna().sum().sum() - missing_before
print("Rows:", len(df), "| missing values created by conversion:", created)
df.dtypes

Rows: 130 | missing values created by conversion: 0


,0
country,object
iso3,object
year,Int16
population_m,float64
gdp_pc_usd,float64
health_spend_pct,float64
life_expectancy,float64
income_group,object
survey_date,object


In [59]:
# Self-check for checkpoint 2
check(len(df) == 130, "130 rows should remain: only the 'Total' row is removed here.")
check(created == 0, "Conversion created missing values: a marker or a format is still undeclared.")
check(all(pd.api.types.is_numeric_dtype(df[c]) for c in ["year", "population_m", "gdp_pc_usd", "health_spend_pct", "life_expectancy"]),
      "Every numeric column should now be numeric.")

✅
✅
✅


**Why count what conversion created?** `errors="coerce"` silently turns anything it cannot read into a missing value. If you declared every marker first, conversion should create **none**: any it creates is a value you have not looked at.

### ✍️ Decision log, checkpoint 2: structure and types

**Driver:** Anissa · **Navigator(s):** Others

| What we found (with a count) | What we decided | Justification (one line) |
|---|---|---|
| | | |

---
## Checkpoint 3: text and categories  🟡 skeleton

Each country, and each income group, must be written **one format only**. First look at the variants, then harmonise.

In [60]:
# 🟢 Look first: every spelling of each country, and every income group label
spellings = df.groupby(df["iso3"].str.upper())["country"].unique()
print(spellings[spellings.map(len) > 1])
print()
print(df["income_group"].value_counts())

iso3
CIV    [CÃ´te d'Ivoire, Côte d'Ivoire, Côte d’Ivoire]
GBR              [United Kingdom, UK, united kingdom]
KOR                        [Korea, Rep., South Korea]
TUR                       [Türkiye, Turkey, Türkiye]
Name: country, dtype: object

income_group
High income            41
Lower middle income    30
Upper middle income    23
Low income             12
low income              6
Lower middle            4
Low                     3
lower-middle income     3
High                    2
high income             2
upper middle income     2
High Income             1
Upper-middle            1
Name: count, dtype: int64


**What to look for:** country names written several ways, including **encoding damage** (`Ã` followed by another character), a **former name**, a **curly apostrophe**, and two names that look identical but are stored differently (**Unicode normalisation**: compare with `unicodedata.is_normalized("NFC", name)`). The ISO3 codes have their own problems: look at their case and spacing. Income groups have several spellings each.

In [61]:
# (a) Standardise the key first: upper-case ISO3 codes (spaces were stripped in checkpoint 2)
df["iso3"] = df["iso3"].str.upper()

# (b) One canonical name per country, from the standard list (World Bank style names)
ISO3_TO_NAME = {"GBR": "United Kingdom", "FRA": "France", "DEU": "Germany", "ITA": "Italy", "ESP": "Spain",
                "USA": "United States", "JPN": "Japan", "KOR": "Korea, Rep.", "QAT": "Qatar", "CHN": "China",
                "BRA": "Brazil", "MEX": "Mexico", "TUR": "Türkiye", "ZAF": "South Africa", "IDN": "Indonesia",
                "IND": "India", "NGA": "Nigeria", "KEN": "Kenya", "CIV": "Côte d'Ivoire", "PAK": "Pakistan",
                "BGD": "Bangladesh", "ETH": "Ethiopia", "UGA": "Uganda", "MOZ": "Mozambique", "NER": "Niger"}
df["country"] = df["iso3"].map(ISO3_TO_NAME)
check(df["country"].notna().all(), "Some ISO3 codes did not map: print df.loc[df['country'].isna(), 'iso3'].")

# (c) Harmonise income groups: lower-case, hyphens to spaces, add "income" where it is missing,
#     then capitalise and store as an ordered category
INCOME = ["Low income", "Lower middle income", "Upper middle income", "High income"]
inc = df["income_group"].str.lower().str.replace("-", " ", regex=False).str.strip()
inc = inc.where(inc.str.endswith("income"), inc + " income")
df["income_group"] = inc.str.capitalize().astype(pd.CategoricalDtype(INCOME, ordered=True))
check(df["income_group"].notna().all(), "A label did not match one of the four categories: print the unmatched ones.")
df["income_group"].value_counts(sort=False)

✅
✅


,count
income_group,
Low income,21
Lower middle income,37
Upper middle income,26
High income,46


In [62]:
# Self-check for checkpoint 3
check(df.groupby("iso3")["country"].nunique().max() == 1, "Each ISO3 code should have exactly one name.")
check(df["country"].map(lambda x: "Ã" not in x and unicodedata.is_normalized("NFC", x)).all(), "Names should be NFC, with no mojibake.")
check(df["income_group"].nunique() == 4, "There should be exactly four income groups.")

✅
✅
✅


✍️ **Think about it before writing the log.** Mapping from ISO3 replaced the damaged names, so you did not have to repair them. If there were no reliable code to map from, how would you have fixed `CÃ´te d'Ivoire` and the decomposed `Türkiye`? (Lecture: encoding damage; Unicode normalisation.)

### ✍️ Decision log, checkpoint 3: text and categories

**Driver:** ____ · **Navigator(s):** ____

| What we found (with a count) | What we decided | Justification (one line) |
|---|---|---|
| | | |

---
## Checkpoint 4: dates  🔵 prompt only

**Your task**: Parse `survey_date` into a **UTC datetime**, and make sure every date means what the source meant.

* Look at the formats present: how many look like `2021-03-15`, and how many like `15/03/2021`? Is any date impossible?
* Parse each format **explicitly** with `format=` (never let a format be guessed: is `05/03/2021` the 5th of March or the 3rd of May?). The header and the other rows tell you which order this source uses.
* Count the dates that could not be parsed. Investigate them; do not lose them silently.

*Methods: `pd.to_datetime(..., format=..., utc=True, errors="coerce")`, combining two parses with a mask, `.isna().sum()`.*

In [63]:
#parsing with YYYY-MM-DD format
date_iso = pd.to_datetime(df["survey_date"], format="%Y-%m-%d", errors="coerce", utc=True)

#then try parsing with DD/MM/YYYY format
date_dmy = pd.to_datetime(df["survey_date"], format="%d/%m/%Y", errors="coerce", utc=True)

#combine and use the YYYY-MM-DD parse and fill in NaNs using the DD/MM/YYYY parse
df["survey_date"] = date_iso.fillna(date_dmy)

#count and then display how many dates have failed to parse
unparsed_count = df["survey_date"].isna().sum()
print(f"Dates that could not be parsed: {unparsed_count}")
if unparsed_count > 0:
    display(df[df["survey_date"].isna()])

Dates that could not be parsed: 1


,country,iso3,year,population_m,gdp_pc_usd,health_spend_pct,life_expectancy,income_group,survey_date
34,Kenya,KEN,2020,52.3,1630.0,NaN,66.0,Lower middle income,NaT


In [64]:
# Self-check for checkpoint 4
check(isinstance(df["survey_date"].dtype, pd.DatetimeTZDtype) and str(df["survey_date"].dt.tz) == "UTC", "survey_date should be a UTC datetime.")
check(df["survey_date"].isna().sum() <= 1, "At most one date should be unparseable: the rest have one of two formats.")

✅
✅


### ✍️ Decision log, checkpoint 4: dates

**Driver:** ____ · **Navigator(s):** ____

| What we found (with a count) | What we decided | Justification (one line) |
|---|---|---|
| | | |

---
## Checkpoint 5: duplicates  🔵 prompt only

**Your task**: Each country–year must appear **exactly once**, and it must be the **right version**.

* Count **exact** duplicate rows, and remove them.
* Count duplicates **by key** (`iso3`, `year`) that remain. Look at them side by side (`keep=False`): what differs between the two versions? The file header mentions **revised records**: which version should you keep, and how can you tell from the data?
* Check the row count against what checkpoint 1 told you the source published.

*Methods: `duplicated()`, `duplicated(subset=..., keep=False)`, `sort_values`, `drop_duplicates(subset=..., keep=...)`.*

In [65]:
# Your code here

In [66]:
# Self-check for checkpoint 5
check(not df.duplicated(["iso3", "year"]).any(), "Each (iso3, year) should appear once.")
check(len(df) == 125, "125 country-years should remain, as the source published.")

❌ Each (iso3, year) should appear once.
❌ 125 country-years should remain, as the source published.


### ✍️ Decision log, checkpoint 5: duplicates

**Driver:** ____ · **Navigator(s):** ____

| What we found (with a count) | What we decided | Justification (one line) |
|---|---|---|
| | | |

---
## Checkpoint 6: units and scales  🔵 prompt only

**Your task**: Every column must be in **one unit**: the units the file header states.

* For each numeric column, compare countries: does any country's values sit at a very different **order of magnitude** from comparable countries, in every year? (Compare like with like: a large country can have a large population.)
* For each problem, decide the conversion from the header's units, apply it **once**, and check the result looks plausible next to similar countries.

*Methods: `groupby("iso3")[col].median()`, `np.log10`, sorting, `.loc[condition, col] *= ...`.*

In [67]:
# Your code here

In [68]:
# Self-check for checkpoint 6
med = df.groupby("iso3")[["population_m", "gdp_pc_usd"]].median()
check(med["population_m"].max() < 2000 and med["gdp_pc_usd"].min() > 100, "A country is still in the wrong unit.")

❌ A country is still in the wrong unit.


### ✍️ Decision log, checkpoint 6: units and scales

**Driver:** ____ · **Navigator(s):** ____

| What we found (with a count) | What we decided | Justification (one line) |
|---|---|---|
| | | |

---
## Checkpoint 7: impossible and implausible values  🔵 prompt only

**Your task**: Every value must be **possible**, and **plausible in context**.

* Write a range rule for each numeric column (what is possible for a life expectancy? a GDP per person?), and list the rows that break it.
* For each one, decide: **correct it** (only if the correct value is evident: what error could have produced it?), **set it missing**, or **keep it** because it is genuinely extreme. Compare with the same country's other years.
* Watch for a genuinely extreme value that a careless rule would remove.

*Methods: `between`, boolean masks, `groupby` and comparison with neighbouring years, `.loc[mask, col] = np.nan`.*

In [69]:
# Your code here

In [70]:
# Self-check for checkpoint 7
check(df["life_expectancy"].dropna().between(30, 95).all(), "Life expectancy should be between 30 and 95 years.")
check((df["gdp_pc_usd"].dropna() > 0).all(), "GDP per person must be positive.")
check((df["iso3"] == "QAT").sum() == 5, "Qatar should still have all 5 years: extreme is not the same as wrong.")

❌ Life expectancy should be between 30 and 95 years.
❌ GDP per person must be positive.
✅


### ✍️ Decision log, checkpoint 7: impossible and implausible values

**Driver:** ____ · **Navigator(s):** ____

| What we found (with a count) | What we decided | Justification (one line) |
|---|---|---|
| | | |

---
## Checkpoint 8: missing values  🔵 prompt only · ⚖️ **the contested checkpoint**

**Your task**: Describe what is missing, judge why, and decide what to do.

1. Count missing values in every column, as a number and a share.
2. For **`health_spend_pct`**, compare the share missing **by income group**. What does the pattern suggest about the mechanism: MCAR, MAR or MNAR?
3. **Decide, as a group, between two defensible options:**
   * **Drop the variable.** About a quarter of its values are missing, unevenly across groups.
   * **Replace missing values by imputation**, such as with the median **within each income group**. You can also add an indicator column recording which values were imputed.
4. Decide what to do with any other missing values (population, life expectancy, the unparsed date), even if the answer is "keep them missing".

**Both options score the same.** You are scored on the **argument** in your log (12 points): the evidence you use, whether you treat the other option fairly, and the consequence of your choice for an analysis. Prepare one minute for the debate.

*Methods: `isna().mean()`, `groupby(...)[col].apply(lambda s: s.isna().mean())`, `drop(columns=...)`, `groupby(...).transform("median")`, `fillna`.*

In [71]:
# Your code here
DECISION = "____"      # "drop" or "impute"

In [72]:
# Self-check for checkpoint 8 (accepts either decision)
if "health_spend_pct" in df:
    flags = [c for c in df.columns if "missing" in c or "imputed" in c]
    check(df["health_spend_pct"].notna().all() and len(flags) >= 1, "Impute every value AND keep an indicator column of what was imputed.")
else:
    print("✅ health_spend_pct dropped: make sure your log argues for it.")

❌ Impute every value AND keep an indicator column of what was imputed.


### ✍️ Decision log, checkpoint 8: missing values ⚖️

**Driver:** ____ · **Navigator(s):** ____

| What we found (with a count) | What we decided | Justification (one line) |
|---|---|---|
| | | |

**Write your argument about the contested checkpoint (at most five sentences)**: State the evidence (share missing, pattern by income group, likely mechanism); say what the **other** option would have gained; state the consequence of your choice for an analysis that uses this variable.

*Our argument:*

---
## Checkpoint 9: validate and save  🔵 prompt only

**Your task**: Write a `validate(d)` function that **fails loudly** (`assert` with a clear message), run it, and save the clean data. "Failing loudly" means causing errors in the code so that people easily notice the problem.

* Schema: the columns you expect, in order, with the dtypes you chose.
* Key: `(iso3, year)` unique; 125 rows.
* Ranges: one range rule per numeric column, as decided in checkpoint 7.
* Relationships: four income groups; every ISO3 code with one name; no date after today.
* Test it: run `validate` once on a deliberately broken copy (for example, add a duplicate row) and confirm it fails.
* Save to `data/processed/{GROUP}_clean.csv` with `index=False`. This is the file the tutor scores.
* **A CSV stores only text.** Your dtypes are not saved: whoever reads the file must re-type it. So write dates in **ISO 8601 with their UTC offset** (pandas does this for a UTC datetime column), and check what comes back when you read your own file.

*Methods: `assert`, `pd.Timestamp.now(tz="UTC")`, `to_csv(..., index=False)`, `read_csv`.*

In [73]:
# Your code here

In [74]:
# Self-check for checkpoint 9
saved = Path(f"data/processed/{GROUP}_clean.csv")
check(saved.exists(), "Save your clean data to data/processed/<GROUP>_clean.csv.")
if saved.exists():
    back = pd.read_csv(saved)
    print(back.dtypes)                      # reading a CSV: survey_date is text again, categories are plain text
    check(back["survey_date"].dropna().astype(str).str.endswith("+00:00").all(), "Write survey_date as a UTC datetime so the file records +00:00.")
    check(len(back) == 125 and not back.duplicated(["iso3", "year"]).any(), "The saved file should have 125 unique country-years.")

❌ Save your clean data to data/processed/<GROUP>_clean.csv.


### ✍️ Decision log, checkpoint 9: validate and save

**Driver:** ____ · **Navigator(s):** ____

| What we found (with a count) | What we decided | Justification (one line) |
|---|---|---|
| | | |

---
## Hand-in: commit, push and open a pull request (5 minutes)

1. **Runtime → Restart and run all.** Ensure that every cell in your notebook must run from the top. (Your reviewers will do exactly this.)
2. Upload this notebook into your group repository; add `data/processed/<GROUP>_clean.csv`. **Do not** commit the raw (original) file: `week4_country_panel_raw.csv`.
3. Share your repo with the reviewing group. Also share it with your tutor.

The tutor's rotation: A reviews B, B reviews C, and so on, round the room.

The tutor scores your saved file during the review round.

---
## Round 2: review another group's decisions (10 minutes)

Work in your own copy of this notebook. **Rotate roles again:** a new driver opens the other group's pull request; the navigators score.

1. **Re-run their notebook** in Colab: open it from their branch, *Restart and run all*. Does it run from top to bottom? Does it produce 125 rows? (4 points)
2. **Score their decision log** with the rubric below (this becomes their "Decision log" score, moderated by the tutor).
3. **Bug bounty:** for each problem in their **clean data** that they missed or got wrong, write one line of evidence (a row, a value, a count). Each one confirmed by the answer key earns **you** 3 points (up to 12).
4. **Comment on their pull request:** at least three specific, constructive comments on lines of code or log entries (4 points). Be specific and kind: "Checkpoint 6: JPN gdp_pc_usd is about 40 in every year: thousands of USD?" beats "units wrong".

### Rubric for their decision log (30 points)

| Item | Points |
|---|---|
| Each of checkpoints 1 to 9: a count, a decision and a one-line justification for every problem found (2 complete, 1 partial, 0 missing) | 9 × 2 = 18 |
| Checkpoint 8 argument: **evidence** used (share missing, pattern by group, mechanism) | 4 |
| Checkpoint 8 argument: the **other option** treated fairly | 4 |
| Checkpoint 8 argument: the **consequence** for an analysis stated | 4 |

In [75]:
# 🟡 Fill in your review, then run: it writes a file to commit to YOUR repository
REVIEWED_GROUP = "____"
log_scores = {"cp1": 0, "cp2": 0, "cp3": 0, "cp4": 0, "cp5": 0, "cp6": 0, "cp7": 0, "cp8": 0, "cp9": 0}   # 0, 1 or 2 each
argument = {"evidence": 0, "other_option": 0, "consequence": 0}                                         # 0 to 4 each
reran_from_top = None          # add True if the notebook run or False if it didn't
rerun_note = "____"            # one line: what happened when you ran it. It it runs, you can just add "It run", otherwise "It did not run"
bounties = [                   # one line of evidence per problem they missed, e.g.
    # "cp6: JPN gdp_pc_usd about 40 in every year: still thousands of USD",
]
pr_comments_made = 0           # how many comments you left on their pull request

review = {"reviewer": GROUP, "reviewed": REVIEWED_GROUP, "log_scores": log_scores, "argument": argument,
          "log_total": sum(log_scores.values()) + sum(argument.values()), "reran_from_top": reran_from_top,
          "rerun_note": rerun_note, "bounties": bounties, "pr_comments_made": pr_comments_made}
Path("reviews").mkdir(exist_ok=True)
Path(f"reviews/{GROUP}_reviews_{REVIEWED_GROUP}.json").write_text(json.dumps(review, indent=2))
print(json.dumps(review, indent=2))

{
  "reviewer": "Group D",
  "reviewed": "____",
  "log_scores": {
    "cp1": 0,
    "cp2": 0,
    "cp3": 0,
    "cp4": 0,
    "cp5": 0,
    "cp6": 0,
    "cp7": 0,
    "cp8": 0,
    "cp9": 0
  },
  "argument": {
    "evidence": 0,
    "other_option": 0,
    "consequence": 0
  },
  "log_total": 0,
  "reran_from_top": null,
  "rerun_note": "____",
  "bounties": [],
  "pr_comments_made": 0
}


Commit `reviews/<GROUP>_reviews_<REVIEWED>.json` to **your** repository and push. The tutor collects every review file for the leaderboard.

---
## Round 3: the debate (one minute per group)

One speaker per group (someone who has not spoken yet today) argues your checkpoint 8 decision in **one minute**: the evidence, why not the other option, the consequence. The class votes on Mentimeter for the **best argument**, whichever side it is on. The winner takes the "Best argument" prize.

✍️ *Speaker:* ____  · *Our one-minute outline (three bullet points):*

---
## Before you leave

- [ ] Every checkpoint log has a count, a decision and a one-line justification.
- [ ] Your notebook runs from the top; your clean CSV file and review file are pushed.
- [ ] You responded to your reviewers' comments on your pull request (before next week).